# Claude Code From Scratch — Single Notebook, Local Ollama

A consolidated reimplementation of the [claude-code-from-scratch](https://github.com/FareedKhan-dev/claude-code-from-scratch) harness, rebuilt against **local Ollama (qwen3)** instead of the Anthropic API.

This notebook condenses sessions s01–s07 of the source repo into one runnable pipeline:

| Source session | Concept folded in here |
|---|---|
| s01 perception-action loop | `agent_loop` |
| s02 tool dispatch map | `TOOLS`, `DISPATCH` |
| s03 todo planning | `todo_write/read/update` |
| s04 subagent isolation | `spawn_subagent` |
| s05 skill loading | `list_skills`, `load_skill` (optional dir) |
| s06 context compaction | `maybe_compress` |
| s07 task graph | `task_create/list/update/next` |

### Model choice per role

Qwen3 is available locally in two sizes. We assign each role the model that matches its job:

| Role | Model | Why |
|---|---|---|
| **Lead agent** (planning, tool orchestration) | `qwen3:32b` | Needs strong reasoning + reliable tool calling |
| **Subagent** (focused, throwaway context) | `qwen3:8b` | Smaller scope, want faster turns |
| **Summarizer** (context compaction) | `qwen3:8b` | Pure condensation task, no tools |

You can override any of these in the config cell.

### Test interface

Cell near the bottom exposes `chat(query)` — call it from another cell to drive the agent. Every step (API call, tool call, tool result, compaction, subagent spawn) is logged so you can see exactly what the agent is doing.


In [ ]:
"""
Imports + logging.

All logging goes through a single logger named "agent". Every subsystem
(agent loop, tools, subagent, compactor, ollama client) uses a child of
this logger so you can adjust verbosity per component if you want.

Default level is INFO — high-signal events only. Flip AGENT_LOG_LEVEL
to "DEBUG" to see full message payloads (verbose).
"""
from __future__ import annotations

import json
import logging
import os
import subprocess
import sys
import time
import uuid
import glob as _glob
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

import requests  # only external dep — used to talk to Ollama HTTP API


# --- Logging setup -----------------------------------------------------------
AGENT_LOG_LEVEL = os.environ.get("AGENT_LOG_LEVEL", "INFO").upper()

# Custom formatter: compact, color-coded by level, shows the subcomponent.
class _Fmt(logging.Formatter):
    COLORS = {
        "DEBUG":    "\033[90m",   # gray
        "INFO":     "\033[36m",   # cyan
        "WARNING":  "\033[33m",   # yellow
        "ERROR":    "\033[31m",   # red
        "CRITICAL": "\033[41m",   # red bg
    }
    RESET = "\033[0m"
    def format(self, record: logging.LogRecord) -> str:
        color = self.COLORS.get(record.levelname, "")
        # record.name is e.g. "agent.tool.bash" — strip the leading "agent."
        short = record.name.split(".", 1)[1] if "." in record.name else record.name
        return f"{color}[{record.levelname:5s}] {short:18s} | {record.getMessage()}{self.RESET}"

_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(_Fmt())

log = logging.getLogger("agent")
log.handlers.clear()
log.addHandler(_handler)
log.setLevel(AGENT_LOG_LEVEL)
log.propagate = False  # don't duplicate into Jupyter's root logger

# Convenience: child loggers per subsystem so we can tag every line.
log_loop    = logging.getLogger("agent.loop")
log_ollama  = logging.getLogger("agent.ollama")
log_tool    = logging.getLogger("agent.tool")
log_sub     = logging.getLogger("agent.subagent")
log_todo    = logging.getLogger("agent.todo")
log_compact = logging.getLogger("agent.compact")
log_chat    = logging.getLogger("agent.chat")

log.info(f"Logger initialised at level {AGENT_LOG_LEVEL}")


In [ ]:
"""
Configuration.

All knobs live here so the rest of the notebook stays declarative.
Edit OLLAMA_HOST / MODELS / WORKDIR if your setup differs.
"""

# --- Ollama HTTP endpoint ----------------------------------------------------
# This machine runs ollama on :8080 (see /etc/systemd/system/ollama.service).
# Change to http://localhost:11434 if you use the default port.
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:8080")

# --- Model per agent role ----------------------------------------------------
# Lead model handles user-facing turns: planning, deciding which tool to call,
# integrating tool results. Tool calling here needs to be reliable, so we use
# the larger qwen3.
#
# Worker (subagent) model runs in an isolated, short-lived context for one
# focused subtask. Lower stakes -> smaller model is fine and faster.
#
# Summariser model only condenses old turns into a paragraph. No tool calls,
# no reasoning chain -> smallest model is enough.
MODELS = {
    "lead":       os.environ.get("AGENT_LEAD_MODEL",       "qwen3:32b"),
    "subagent":   os.environ.get("AGENT_SUBAGENT_MODEL",   "qwen3:8b"),
    "summarizer": os.environ.get("AGENT_SUMMARIZER_MODEL", "qwen3:8b"),
}

# --- Sandbox / working dir ---------------------------------------------------
# All tools (bash, read, write, grep, glob) operate relative to WORKDIR.
# Default to a sandbox under the notebook dir so we don't accidentally touch
# unrelated files. Change to "/home/bmartins/dev/some_project" to point the
# agent at a real codebase.
WORKDIR = Path(os.environ.get("AGENT_WORKDIR",
                              str(Path.cwd() / "sandbox"))).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)

# --- Optional skills directory -----------------------------------------------
SKILLS_DIR = Path(os.environ.get("AGENT_SKILLS_DIR",
                                 str(WORKDIR / "skills")))

# --- State files (persisted across runs) -------------------------------------
TODO_FILE   = WORKDIR / ".agent_todo.json"
TASKS_FILE  = WORKDIR / ".agent_tasks.json"
MEMORY_FILE = WORKDIR / ".agent_memory.md"

# --- Limits ------------------------------------------------------------------
MAX_TOOL_OUTPUT     = 20_000   # truncate big tool outputs to keep ctx lean
MAX_TURNS           = 25       # hard cap on agent loop iterations per query
COMPRESS_THRESHOLD  = 40_000   # chars across all messages -> trigger compact
KEEP_RECENT         = 6        # last N messages preserved verbatim during compact
BASH_TIMEOUT_S      = 60       # subprocess timeout
REQUEST_TIMEOUT_S   = 600      # ollama HTTP timeout (32b can be slow on big ctx)

# --- Bash blocklist ----------------------------------------------------------
BASH_BLOCKLIST = [
    "rm -rf /", "sudo", "shutdown", "reboot", "mkfs",
    "> /dev/", ":(){ :|:& };:",
]

log.info(f"OLLAMA_HOST = {OLLAMA_HOST}")
log.info(f"WORKDIR     = {WORKDIR}")
log.info(f"MODELS      = {MODELS}")


In [ ]:
"""
Ollama client.

We hit Ollama's native /api/chat endpoint with stream=False. This endpoint
supports OpenAI-style tool calling: pass `tools=[...]`, and the model can
return `message.tool_calls=[{function: {name, arguments}}]`.

We deliberately keep this layer thin -- no retry, no streaming. Errors
bubble up as exceptions so the loop can decide what to do.
"""

def ollama_chat(
    *,
    model: str,
    messages: List[Dict[str, Any]],
    tools: Optional[List[Dict[str, Any]]] = None,
    temperature: float = 0.2,
) -> Dict[str, Any]:
    """
    Call Ollama /api/chat. Returns the raw `message` dict from the response,
    which looks like:
        {"role": "assistant", "content": "...",
         "tool_calls": [{"function": {"name": "...", "arguments": {...}}}]}
    """
    payload: Dict[str, Any] = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature},
    }
    if tools:
        payload["tools"] = tools

    n_msgs = len(messages)
    approx_chars = sum(len(str(m.get("content", ""))) for m in messages)
    log_ollama.info(
        f"-> {model}  msgs={n_msgs}  ~{approx_chars} chars  "
        f"tools={len(tools) if tools else 0}"
    )
    log_ollama.debug(f"   request last msg: {str(messages[-1])[:300]}")

    t0 = time.time()
    resp = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=REQUEST_TIMEOUT_S,
    )
    dt = time.time() - t0

    if resp.status_code != 200:
        log_ollama.error(f"<- HTTP {resp.status_code}: {resp.text[:500]}")
        resp.raise_for_status()

    data = resp.json()
    msg = data.get("message", {})
    tcs = msg.get("tool_calls") or []
    text_len = len(msg.get("content") or "")

    log_ollama.info(
        f"<- {model}  {dt:5.1f}s  text={text_len}ch  "
        f"tool_calls={len(tcs)}  done_reason={data.get('done_reason','?')}"
    )
    log_ollama.debug(f"   response message: {str(msg)[:500]}")
    return msg


def ollama_healthcheck() -> bool:
    """Quick sanity probe -- confirms the server is up and lists models."""
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
        r.raise_for_status()
        tags = [m["name"] for m in r.json().get("models", [])]
        log_ollama.info(f"healthcheck OK -- {len(tags)} models available")
        # Verify the models we declared are actually pulled.
        for role, name in MODELS.items():
            present = any(t == name or t.startswith(name + ":") for t in tags)
            mark = "OK" if present else "MISSING"
            log_ollama.info(f"   {role:11s} {name:18s} [{mark}]")
        return True
    except Exception as e:
        log_ollama.error(f"healthcheck FAILED: {e}")
        return False

ollama_healthcheck()


In [ ]:
"""
Core tools.

These are the building blocks the agent uses to observe and change the
world. Each tool is a plain Python function returning a string -- so any
output, including errors, ends up back in the conversation as a tool result.

Safety:
  - All file paths are resolved against WORKDIR and rejected if they
    escape it (no `../../etc/passwd`).
  - bash() refuses anything matching BASH_BLOCKLIST and runs in WORKDIR.
  - All outputs are truncated to MAX_TOOL_OUTPUT to protect context.

Undo:
  - write() snapshots the previous content (or marks the file as
    newly-created) into SNAPSHOTS so revert() can roll it back.
"""

# In-memory undo stack: {abs_path: previous_content or None}
SNAPSHOTS: Dict[str, Optional[str]] = {}


def _safe_path(path: str) -> Path:
    """Resolve `path` and ensure it lives under WORKDIR. Raises ValueError if not."""
    p = (WORKDIR / path).resolve() if not os.path.isabs(path) else Path(path).resolve()
    try:
        p.relative_to(WORKDIR)
    except ValueError:
        raise ValueError(f"path escapes WORKDIR: {p}")
    return p


def _truncate(s: str, limit: int = MAX_TOOL_OUTPUT) -> str:
    if len(s) <= limit:
        return s
    return s[:limit] + f"\n... [truncated {len(s) - limit} chars]"


# --- bash --------------------------------------------------------------------
def tool_bash(command: str) -> str:
    log_tool.info(f"[bash] {command[:120]}")
    for bad in BASH_BLOCKLIST:
        if bad in command:
            log_tool.warning(f"[bash] BLOCKED ({bad!r}): {command[:80]}")
            return f"Error: blocked by safety policy (matched {bad!r})"
    try:
        proc = subprocess.run(
            command, shell=True, cwd=str(WORKDIR),
            capture_output=True, text=True, timeout=BASH_TIMEOUT_S,
        )
        out = (proc.stdout + proc.stderr).strip() or "(no output)"
        log_tool.info(f"[bash] exit={proc.returncode} bytes={len(out)}")
        log_tool.debug(f"[bash] output: {out[:300]}")
        return _truncate(out)
    except subprocess.TimeoutExpired:
        log_tool.error(f"[bash] timeout after {BASH_TIMEOUT_S}s")
        return f"Error: timeout after {BASH_TIMEOUT_S}s"
    except Exception as e:
        log_tool.error(f"[bash] exception: {e}")
        return f"Error: {e}"


# --- read --------------------------------------------------------------------
def tool_read(path: str, start_line: Optional[int] = None, end_line: Optional[int] = None) -> str:
    log_tool.info(f"[read] {path} lines={start_line}:{end_line}")
    try:
        p = _safe_path(path)
        lines = p.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)
        i0 = max(0, (start_line or 1) - 1)
        i1 = end_line if end_line is not None else len(lines)
        numbered = "".join(f"{i0+1+i:5d}\t{ln}" for i, ln in enumerate(lines[i0:i1]))
        out = numbered or "(empty)"
        log_tool.debug(f"[read] {len(lines)} total lines, returning {i1-i0}")
        return _truncate(out)
    except FileNotFoundError:
        log_tool.warning(f"[read] not found: {path}")
        return f"Error: file not found: {path}"
    except Exception as e:
        log_tool.error(f"[read] {e}")
        return f"Error: {e}"


# --- write -------------------------------------------------------------------
def tool_write(path: str, content: str) -> str:
    log_tool.info(f"[write] {path} ({len(content)} chars)")
    try:
        p = _safe_path(path)
        # Snapshot previous state for revert()
        if p.exists():
            SNAPSHOTS[str(p)] = p.read_text(encoding="utf-8", errors="replace")
            action = "updated"
        else:
            SNAPSHOTS[str(p)] = None
            action = "created"
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding="utf-8")
        log_tool.info(f"[write] {action} {p} (snapshot saved)")
        return f"{action}: {p} (snapshot saved -- use revert to undo)"
    except Exception as e:
        log_tool.error(f"[write] {e}")
        return f"Error: {e}"


# --- grep --------------------------------------------------------------------
def tool_grep(pattern: str, path: str = ".", recursive: bool = True) -> str:
    log_tool.info(f"[grep] {pattern!r} in {path} recursive={recursive}")
    try:
        p = _safe_path(path)
        flags = ["-rn"] if recursive else ["-n"]
        proc = subprocess.run(
            ["grep", *flags, pattern, str(p)],
            capture_output=True, text=True, timeout=30,
        )
        out = (proc.stdout + proc.stderr).strip() or "(no matches)"
        log_tool.debug(f"[grep] {len(out.splitlines())} matches")
        return _truncate(out, 10_000)
    except subprocess.TimeoutExpired:
        return "Error: grep timeout"
    except Exception as e:
        log_tool.error(f"[grep] {e}")
        return f"Error: {e}"


# --- glob --------------------------------------------------------------------
def tool_glob(pattern: str) -> str:
    log_tool.info(f"[glob] {pattern}")
    # Make pattern relative to WORKDIR
    full = str(WORKDIR / pattern) if not os.path.isabs(pattern) else pattern
    matches = sorted(_glob.glob(full, recursive=True))[:200]
    # Filter to keep only paths inside WORKDIR (defensive)
    safe_matches = [m for m in matches if Path(m).resolve().is_relative_to(WORKDIR)]
    log_tool.debug(f"[glob] {len(safe_matches)} matches")
    return "\n".join(safe_matches) if safe_matches else "(no matches)"


# --- revert ------------------------------------------------------------------
def tool_revert(path: str) -> str:
    log_tool.info(f"[revert] {path}")
    try:
        p = _safe_path(path)
        key = str(p)
        if key not in SNAPSHOTS:
            log_tool.warning(f"[revert] no snapshot for {p}")
            return f"Error: no snapshot for {p}"
        prev = SNAPSHOTS.pop(key)
        if prev is None:
            p.unlink(missing_ok=True)
            return f"reverted: deleted {p} (was a new file)"
        p.write_text(prev, encoding="utf-8")
        return f"reverted: {p}"
    except Exception as e:
        log_tool.error(f"[revert] {e}")
        return f"Error: {e}"

log.info("Core tools defined: bash, read, write, grep, glob, revert")


In [ ]:
"""
Tool schemas + dispatch map.

Ollama expects tools in OpenAI's function-calling format:

    {
      "type": "function",
      "function": {
        "name": "...",
        "description": "...",
        "parameters": <JSON schema>
      }
    }

The DISPATCH map routes a tool name (what the model returns in
`tool_calls[i].function.name`) to a Python callable that takes the
arguments dict and returns the string result.

Two registries:
  TOOLS_BASE / DISPATCH_BASE -- file & shell tools (this cell).
  Later cells extend these with todo_*, task_*, spawn_subagent,
  list_skills/load_skill -- so the lead agent gets the full set,
  while subagents can be given a smaller subset on purpose.
"""

def _fn(name: str, description: str, properties: Dict[str, Any], required: List[str]) -> Dict[str, Any]:
    """Helper to build one tool-schema entry without copy-paste boilerplate."""
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }


TOOLS_BASE: List[Dict[str, Any]] = [
    _fn(
        "bash",
        "Run a shell command inside the sandbox WORKDIR. Stdout+stderr are returned.",
        {"command": {"type": "string", "description": "Shell command to execute."}},
        ["command"],
    ),
    _fn(
        "read",
        "Read a text file (relative to WORKDIR). Optional 1-indexed line range.",
        {
            "path":       {"type": "string"},
            "start_line": {"type": "integer", "description": "First line (1-indexed)."},
            "end_line":   {"type": "integer", "description": "Last line (inclusive)."},
        },
        ["path"],
    ),
    _fn(
        "write",
        "Write content to a file. Previous content is snapshotted; use revert to undo.",
        {
            "path":    {"type": "string"},
            "content": {"type": "string"},
        },
        ["path", "content"],
    ),
    _fn(
        "grep",
        "Regex search across files under a path.",
        {
            "pattern":   {"type": "string"},
            "path":      {"type": "string", "description": "Defaults to '.'"},
            "recursive": {"type": "boolean"},
        },
        ["pattern"],
    ),
    _fn(
        "glob",
        "Find files matching a glob pattern, e.g. '**/*.py'.",
        {"pattern": {"type": "string"}},
        ["pattern"],
    ),
    _fn(
        "revert",
        "Restore a file to its state before the most recent write.",
        {"path": {"type": "string"}},
        ["path"],
    ),
]

# Map tool name -> (callable taking the args dict, returning str result).
# Each lambda unpacks/defaults the args so the underlying function gets
# exactly what it expects.
DISPATCH_BASE: Dict[str, Callable[[Dict[str, Any]], str]] = {
    "bash":   lambda a: tool_bash(a["command"]),
    "read":   lambda a: tool_read(a["path"], a.get("start_line"), a.get("end_line")),
    "write":  lambda a: tool_write(a["path"], a["content"]),
    "grep":   lambda a: tool_grep(a["pattern"], a.get("path", "."), a.get("recursive", True)),
    "glob":   lambda a: tool_glob(a["pattern"]),
    "revert": lambda a: tool_revert(a["path"]),
}

log.info(f"Base tool registry: {list(DISPATCH_BASE)}")


In [ ]:
"""
Agent loop -- perception -> action -> observation -> repeat.

This is the heart of the agent. Every Claude-Code-style harness is, at its
core, a `while True` that does three things:

  1. PERCEPTION  : send the conversation so far to the model
  2. ACTION      : if the model emitted tool_calls, run them
  3. OBSERVATION : feed the tool results back as new messages

The loop terminates the moment the model returns a message with no
tool_calls -- that final text *is* the answer. MAX_TURNS is a safety cap
so a runaway model can't loop forever.

Why a list of `messages` instead of a string?
  Tool-using models are stateless w.r.t. the conversation. The full
  history (system + user + assistant + tool results) is resent each
  turn. We just keep appending.

Message roles:
  "system"    -- instructions, sent once at the top
  "user"      -- the human's query (or a synthetic prompt from a parent)
  "assistant" -- the model's reply; may carry tool_calls
  "tool"      -- the *result* of running one tool_call (one msg per call)
"""

def _run_one_tool_call(tc: Dict[str, Any], dispatch: Dict[str, Callable]) -> Dict[str, Any]:
    """Execute a single tool_call dict from Ollama; return a 'tool' role message."""
    fn   = tc.get("function", {}) or {}
    name = fn.get("name", "")
    args = fn.get("arguments", {}) or {}
    # Ollama usually returns arguments as a pre-parsed dict, but be defensive.
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            args = {}

    arg_preview = ", ".join(f"{k}={str(v)[:40]!r}" for k, v in args.items())
    log_tool.info(f"-> {name}({arg_preview})")

    if name not in dispatch:
        result = f"[error] Unknown tool: {name!r}. Available: {sorted(dispatch)}"
        log_tool.warning(result)
    else:
        try:
            result = dispatch[name](args)
        except Exception as e:
            # Tool errors are observations, not crashes -- feed them back so
            # the model can self-correct rather than aborting the whole loop.
            result = f"[error] Tool {name!r} raised {type(e).__name__}: {e}"
            log_tool.exception(result)

    result = _truncate(str(result), MAX_TOOL_OUTPUT)
    log_tool.debug(f"<- {name} result ({len(result)} chars):\n{result}")
    return {"role": "tool", "name": name, "content": result}


def agent_loop(
    messages: List[Dict[str, Any]],
    tools:    List[Dict[str, Any]],
    dispatch: Dict[str, Callable],
    model:    str = MODELS["lead"],
    max_turns: int = MAX_TURNS,
    label:    str = "lead",
) -> str:
    """
    Run perception-action until the model stops asking for tools.

    `messages` is mutated in-place -- the caller can inspect the full
    transcript afterwards (assistant turns + tool observations).
    Returns the final assistant text (the answer).
    """
    tool_names = [t["function"]["name"] for t in tools]
    log_loop.info(f"[{label}] starting (model={model}, max_turns={max_turns}, tools={tool_names})")

    final_text = ""
    for turn in range(1, max_turns + 1):
        log_loop.info(f"[{label}] turn {turn}/{max_turns}  msgs={len(messages)}")

        # ---- PERCEPTION ----------------------------------------------------
        msg = ollama_chat(model=model, messages=messages, tools=tools)
        # msg looks like: {"role": "assistant", "content": "...", "tool_calls": [...]}
        messages.append(msg)

        tool_calls = msg.get("tool_calls") or []
        text       = (msg.get("content") or "").strip()

        # ---- TERMINATION ---------------------------------------------------
        if not tool_calls:
            final_text = text
            preview = final_text[:120] + ("..." if len(final_text) > 120 else "")
            log_loop.info(f"[{label}] DONE on turn {turn} -- no tool_calls. final={preview!r}")
            return final_text

        # ---- ACTION + OBSERVATION ------------------------------------------
        log_loop.info(f"[{label}] turn {turn}: model requested {len(tool_calls)} tool call(s)")
        if text:
            # Some models emit reasoning text alongside tool_calls.
            log_loop.debug(f"[{label}] assistant said (alongside tool_calls): {text[:200]}")

        for tc in tool_calls:
            messages.append(_run_one_tool_call(tc, dispatch))

    log_loop.warning(f"[{label}] hit MAX_TURNS={max_turns} without termination.")
    return final_text or "[agent_loop] max turns reached without a terminal response"


log.info("Agent loop ready: agent_loop(messages, tools, dispatch, model=..., label=...)")


In [ ]:
"""
Subagents -- isolated context for messy subtasks.

When the lead model hits a task that would pollute its context (deep
exploration, many failed grep attempts, reading a giant file just to
find one symbol) it can call `spawn_subagent(prompt=...)`. That spawns
a fresh `agent_loop` with:

  - an empty messages list seeded only with the subtask prompt
  - the smaller subagent model (qwen3:8b -- faster, scope is narrow)
  - the BASE tool set (no spawn_subagent itself -> no infinite recursion)

The subagent runs to completion; only its final text string returns
to the lead. The lead never sees the subagent's intermediate turns,
which is the whole point.

State mapping:
  Lead:     [..., user("explore X"), asst(spawn_subagent(...)), tool(<subagent result>), ...]
  Subagent: [system, user("explore X"), asst(...), tool(...), ..., asst("Found it: ...")]
"""

SUBAGENT_SYSTEM = (
    f"You are a focused subagent working in a sandbox at {WORKDIR}. "
    "You have a single subtask. Use tools to investigate or modify files as needed. "
    "When you are confident in the answer, reply with a concise summary and stop "
    "calling tools -- that final message is what gets returned to the lead agent. "
    "Do not ask clarifying questions; make a reasonable assumption and proceed."
)


def run_subagent(prompt: str) -> str:
    """
    Run a fresh agent_loop with the subagent model on an isolated history.
    Returns the subagent's final text to the caller (which is itself a tool result).
    """
    sub_id = uuid.uuid4().hex[:6]
    log_sub.info(f"[sub:{sub_id}] spawn -- prompt: {prompt[:120]!r}")

    # Fresh, isolated message list. Note: subagent gets BASE tools only.
    sub_messages: List[Dict[str, Any]] = [
        {"role": "system", "content": SUBAGENT_SYSTEM},
        {"role": "user",   "content": prompt},
    ]

    try:
        result = agent_loop(
            messages=sub_messages,
            tools=TOOLS_BASE,
            dispatch=DISPATCH_BASE,
            model=MODELS["subagent"],
            label=f"sub:{sub_id}",
        )
    except Exception as e:
        log_sub.exception(f"[sub:{sub_id}] crashed: {e}")
        return f"[subagent error] {type(e).__name__}: {e}"

    log_sub.info(f"[sub:{sub_id}] done -- {len(result)} chars: "
                 f"{result[:120]!r}{'...' if len(result) > 120 else ''}")
    return result


# --- Register spawn_subagent as a tool the LEAD agent can call ---------------
# Extend (don't mutate) the base registries so subagents keep getting the
# bare set without this tool -- preventing nested-subagent recursion.
TOOLS_LEAD: List[Dict[str, Any]] = TOOLS_BASE + [
    _fn(
        "spawn_subagent",
        ("Delegate a self-contained subtask to a fresh subagent. Use for exploration, "
         "noisy investigations, or anything that would clutter your own context. "
         "Returns only the subagent's final summary."),
        {"prompt": {"type": "string",
                    "description": "Detailed self-contained instructions for the subagent."}},
        ["prompt"],
    ),
]

DISPATCH_LEAD: Dict[str, Callable[[Dict[str, Any]], str]] = {
    **DISPATCH_BASE,
    "spawn_subagent": lambda a: run_subagent(a["prompt"]),
}

log.info(f"Subagent ready. Lead tool registry now: {list(DISPATCH_LEAD)}")


In [ ]:
"""
Todo planning -- the agent's working plan for the *current* request.

A "todo" is a lightweight, ordered checklist the agent maintains for the
multi-step query it's working on right now. Think of it as scratch paper:
written at the start of a complex request, checked off as steps complete,
overwritten on the next request.

Why a tool instead of just letting the model "think out loud"?
  - Forces the model to commit to a concrete plan up front.
  - Gives us (the user) visibility into what the agent intends to do.
  - Survives across model turns even when the conversation grows --
    the model can re-read its own plan with todo_read instead of
    re-deriving it from scratch.

Three tools:
  todo_write(items)          -- replace the whole list. Usual first move.
  todo_read()                -- print the current list with statuses.
  todo_update(index, status) -- mark one item pending / in_progress / done.

State lives in TODO_FILE so it persists across notebook restarts.

Compare with the task graph in the next-next cell: that one is heavier
(UUIDs, dependencies, priorities) and meant for longer-horizon work
spanning multiple chats. Todos are for "right now, this request, these
5 steps".
"""

_TODO_STATUSES = ("pending", "in_progress", "done")


def _load_todos() -> List[Dict[str, Any]]:
    if not TODO_FILE.exists():
        return []
    try:
        return json.loads(TODO_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as e:
        log_todo.warning(f"todo file unreadable, starting empty: {e}")
        return []


def _save_todos(items: List[Dict[str, Any]]) -> None:
    TODO_FILE.write_text(json.dumps(items, indent=2), encoding="utf-8")


def run_todo_write(items: List[str]) -> str:
    """Replace the whole todo list. All new items start as 'pending'."""
    log_todo.info(f"[todo_write] {len(items)} items")
    todos = [{"index": i, "text": t, "status": "pending"} for i, t in enumerate(items)]
    _save_todos(todos)
    log_todo.debug(f"[todo_write] saved -> {TODO_FILE}")
    return f"Wrote {len(todos)} todos:\n" + run_todo_read()


def run_todo_read() -> str:
    """Render the current list with status markers."""
    todos = _load_todos()
    if not todos:
        return "(no todos)"
    marks = {"pending": "[ ]", "in_progress": "[~]", "done": "[x]"}
    lines = [f"{marks.get(t['status'], '[?]')} {t['index']}: {t['text']}" for t in todos]
    out = "\n".join(lines)
    log_todo.debug(f"[todo_read]\n{out}")
    return out


def run_todo_update(index: int, status: str) -> str:
    """Set one todo's status. `index` is 0-based, matching todo_write order."""
    log_todo.info(f"[todo_update] #{index} -> {status}")
    if status not in _TODO_STATUSES:
        return f"Error: status must be one of {_TODO_STATUSES}, got {status!r}"
    todos = _load_todos()
    for t in todos:
        if t["index"] == index:
            t["status"] = status
            _save_todos(todos)
            return f"Todo #{index} -> {status}\n" + run_todo_read()
    return f"Error: no todo with index {index}"


# --- Register on the LEAD registry ------------------------------------------
# We extend TOOLS_LEAD/DISPATCH_LEAD in place so the lead agent gets the
# planning tools. Subagents intentionally don't get them: a subagent has a
# single focused task, no need for its own checklist.
TOOLS_LEAD += [
    _fn(
        "todo_write",
        ("Set the agent's working todo list. Call this at the start of any "
         "multi-step request to lay out your plan. Replaces any existing list."),
        {"items": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Ordered list of plain-text steps.",
        }},
        ["items"],
    ),
    _fn(
        "todo_read",
        "Show the current todo list with statuses.",
        {},
        [],
    ),
    _fn(
        "todo_update",
        "Update the status of one todo. Use as you make progress.",
        {
            "index":  {"type": "integer", "description": "0-based index from todo_write order."},
            "status": {"type": "string",  "enum": list(_TODO_STATUSES)},
        },
        ["index", "status"],
    ),
]

DISPATCH_LEAD.update({
    "todo_write":  lambda a: run_todo_write(a["items"]),
    "todo_read":   lambda a: run_todo_read(),
    "todo_update": lambda a: run_todo_update(a["index"], a["status"]),
})

log.info(f"Todo tools registered. Lead registry: {list(DISPATCH_LEAD)}")


In [ ]:
"""
Context compaction -- making room when the conversation gets long.

LLMs have a finite context window. A long agent session (many tool calls,
big file reads) fills it up; eventually requests get slow, expensive, or
rejected. Claude Code solves this with rolling compression. We mirror that
with three layers:

  Layer 1 (verbatim)  : keep the last KEEP_RECENT messages exactly as-is,
                        so immediate short-term context is never lost.
  Layer 2 (summarise) : everything older gets condensed by the summariser
                        model (qwen3:8b) into one tight paragraph.
  Layer 3 (persist)   : that summary is also written to MEMORY_FILE on disk,
                        so context can survive a notebook restart.

Trigger: we estimate total size as a character count across all messages.
When it crosses COMPRESS_THRESHOLD, we compress. (Chars are a crude proxy
for tokens -- good enough to decide *when* to act.)

The summariser is told to preserve decisions, file paths, and pending
work, and to drop chit-chat -- because the summary becomes the agent's
only memory of those older turns.
"""

def _estimate_size(messages: List[Dict[str, Any]]) -> int:
    """Crude token proxy: total characters of all message content."""
    return sum(len(str(m.get("content", "") or "")) for m in messages)


def _render_for_summary(messages: List[Dict[str, Any]]) -> str:
    """Flatten messages into a plain transcript the summariser can read."""
    parts = []
    for m in messages:
        role = m.get("role", "?")
        content = str(m.get("content", "") or "")
        # Note tool calls explicitly -- they carry intent even with empty content.
        if m.get("tool_calls"):
            names = [tc.get("function", {}).get("name", "?") for tc in m["tool_calls"]]
            content = (content + f"  (called tools: {', '.join(names)})").strip()
        parts.append(f"[{role}] {content}")
    return "\n\n".join(parts)


def _summarize(messages: List[Dict[str, Any]]) -> str:
    """Use the summariser model to condense a slice of history into a paragraph."""
    transcript = _render_for_summary(messages)
    log_compact.info(f"summarising {len(messages)} msgs (~{len(transcript)} chars) "
                     f"with {MODELS['summarizer']}")
    summary_messages = [
        {"role": "system", "content": (
            "You are a context compressor. Condense the conversation below into a "
            "concise summary. PRESERVE: technical decisions, file paths, code changes "
            "made, tool results that matter, and any pending/unfinished work. DROP: "
            "pleasantries and redundant back-and-forth. Write prose, not a list.")},
        {"role": "user", "content": f"Summarise this history:\n\n{transcript[:20000]}"},
    ]
    # No tools here -- pure text condensation.
    msg = ollama_chat(model=MODELS["summarizer"], messages=summary_messages, tools=None)
    summary = (msg.get("content") or "").strip()
    log_compact.info(f"summary produced ({len(summary)} chars)")
    return summary


def maybe_compress(messages: List[Dict[str, Any]]) -> bool:
    """
    If history is over threshold, replace old turns with a single summary,
    in-place. Returns True if compression happened.

    Resulting shape:
        [ system?,                              <- preserved if present
          user("[summary of earlier turns]"),  <- the compressed memory
          assistant("Understood..."),          <- keeps role alternation sane
          <last KEEP_RECENT messages verbatim> ]
    """
    size = _estimate_size(messages)
    if size < COMPRESS_THRESHOLD:
        return False
    if len(messages) <= KEEP_RECENT + 1:
        return False  # nothing meaningful to compress yet

    log_compact.info(f"context ~{size} chars >= {COMPRESS_THRESHOLD} -- compressing")

    # Preserve a leading system message (instructions) if there is one.
    head = []
    body = messages
    if messages and messages[0].get("role") == "system":
        head = [messages[0]]
        body = messages[1:]

    old    = body[:-KEEP_RECENT]
    recent = body[-KEEP_RECENT:]
    if not old:
        return False

    summary = _summarize(old)

    # Layer 3: persist to disk so it survives a kernel restart.
    try:
        MEMORY_FILE.write_text(
            f"# Agent context memory\n\n{summary}\n", encoding="utf-8"
        )
        log_compact.info(f"summary persisted -> {MEMORY_FILE}")
    except OSError as e:
        log_compact.error(f"could not persist memory: {e}")

    # Rebuild the list in-place so the caller's reference stays valid.
    messages.clear()
    messages.extend(head)
    messages.append({"role": "user",
                     "content": f"[Summary of earlier turns]:\n\n{summary}"})
    messages.append({"role": "assistant",
                     "content": "Understood. I've integrated the summary of our earlier progress."})
    messages.extend(recent)

    log_compact.info(f"compressed {len(old)} msgs into 1 summary; "
                     f"history now {len(messages)} msgs")
    return True


log.info("Context compaction ready: maybe_compress(messages)")


In [ ]:
"""
Task graph -- a persistent, dependency-aware backlog.

This is the heavier sibling of the todo list (cell with todo_*). Where a
todo is throwaway scratch paper for the current request, a *task* is a
durable unit of work that:

  - has a stable unique ID (so it can be referenced later / across sessions)
  - can depend on other tasks (a directed acyclic graph, not a flat list)
  - carries a priority and a result field
  - persists to TASKS_FILE and survives restarts

The payoff is task_next: instead of the model eyeballing the list, we
compute the next *unblocked* task -- a pending task whose dependencies are
all done. That turns "what should I do next?" from a judgement call into
a graph query.

Task schema (one dict per task):
  {"id": "a1b2c3d4", "description": "...", "status": "pending",
   "priority": "medium", "depends_on": ["<id>", ...], "result": ""}

Statuses: pending | in_progress | done | failed
"""

_TASK_STATUSES = ("pending", "in_progress", "done", "failed")


def _load_tasks() -> List[Dict[str, Any]]:
    if not TASKS_FILE.exists():
        return []
    try:
        return json.loads(TASKS_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as e:
        log.warning(f"[task] tasks file unreadable, starting empty: {e}")
        return []


def _save_tasks(tasks: List[Dict[str, Any]]) -> None:
    TASKS_FILE.write_text(json.dumps(tasks, indent=2), encoding="utf-8")


def run_task_create(description: str,
                    depends_on: Optional[List[str]] = None,
                    priority: str = "medium") -> str:
    """Add a task to the graph. Returns the new 8-char ID."""
    tasks = _load_tasks()
    task_id = uuid.uuid4().hex[:8]
    tasks.append({
        "id":          task_id,
        "description": description,
        "status":      "pending",
        "priority":    priority,
        "depends_on":  depends_on or [],
        "result":      "",
    })
    _save_tasks(tasks)
    log.info(f"[task] created {task_id}: {description[:60]} (deps={depends_on or []})")
    return f"Created task {task_id}: {description}"


def run_task_list() -> str:
    """Render every task with id, status, priority, deps."""
    tasks = _load_tasks()
    if not tasks:
        return "(no tasks)"
    lines = []
    for t in tasks:
        deps = f" needs={','.join(t['depends_on'])}" if t.get("depends_on") else ""
        lines.append(f"[{t['id']}] {t['status']:11s} {t['priority']:6s}{deps}  {t['description']}")
    out = "\n".join(lines)
    log.debug(f"[task] list ({len(tasks)} tasks)")
    return out


def run_task_update(task_id: str, status: str, result: str = "") -> str:
    """Set a task's status (matched by full id or unique prefix). Optional result."""
    if status not in _TASK_STATUSES:
        return f"Error: status must be one of {_TASK_STATUSES}, got {status!r}"
    tasks = _load_tasks()
    for t in tasks:
        if t["id"].startswith(task_id):
            t["status"] = status
            if result:
                t["result"] = result
            _save_tasks(tasks)
            log.info(f"[task] {t['id']} -> {status}")
            return f"Task {t['id']} -> {status}"
    return f"Error: no task matching id {task_id!r}"


def run_task_next() -> str:
    """
    Return the next actionable task: the first pending task whose every
    dependency is already 'done'. This is the graph's scheduling logic.
    """
    tasks = _load_tasks()
    done_ids = {t["id"] for t in tasks if t["status"] == "done"}
    for t in tasks:
        if t["status"] != "pending":
            continue
        if all(dep in done_ids for dep in t.get("depends_on", [])):
            log.info(f"[task] next -> {t['id']}")
            return f"Next: [{t['id']}] ({t['priority']}) {t['description']}"
    return "No unblocked tasks. Everything is done, in progress, or waiting on a dependency."


# --- Register on the LEAD registry ------------------------------------------
TOOLS_LEAD += [
    _fn(
        "task_create",
        "Create a persistent task in the dependency graph. Use for multi-session or "
        "dependency-ordered work (heavier than todo_write).",
        {
            "description": {"type": "string"},
            "depends_on":  {"type": "array", "items": {"type": "string"},
                            "description": "IDs of tasks that must be done first."},
            "priority":    {"type": "string", "enum": ["high", "medium", "low"]},
        },
        ["description"],
    ),
    _fn(
        "task_list",
        "List all tasks with IDs, status, priority and dependencies.",
        {},
        [],
    ),
    _fn(
        "task_update",
        "Update a task's status (and optionally record its result).",
        {
            "task_id": {"type": "string", "description": "Full ID or unique prefix."},
            "status":  {"type": "string", "enum": list(_TASK_STATUSES)},
            "result":  {"type": "string", "description": "Optional summary of work done."},
        },
        ["task_id", "status"],
    ),
    _fn(
        "task_next",
        "Ask the graph for the next unblocked task (all dependencies satisfied).",
        {},
        [],
    ),
]

DISPATCH_LEAD.update({
    "task_create": lambda a: run_task_create(a["description"], a.get("depends_on"),
                                             a.get("priority", "medium")),
    "task_list":   lambda a: run_task_list(),
    "task_update": lambda a: run_task_update(a["task_id"], a["status"], a.get("result", "")),
    "task_next":   lambda a: run_task_next(),
})

log.info(f"Task graph registered. Lead registry: {list(DISPATCH_LEAD)}")


In [ ]:
"""
Skill loading -- lazy, on-demand domain knowledge.

A "skill" is a folder under SKILLS_DIR containing a SKILL.md file:

    skills/
      pdf-extraction/
        SKILL.md         <- a how-to: instructions, code snippets, gotchas
      git-bisect/
        SKILL.md

The problem skills solve: you might have dozens of domain playbooks, but
dumping all of them into the system prompt every request wastes context
(and confuses the model with irrelevant material). The Claude-Code answer
is lazy loading:

  1. The agent's system prompt gets only a one-line INDEX of skill names
     + their first-line descriptions (cheap -- a few hundred chars).
  2. When the model decides a skill is relevant, it calls load_skill(name)
     to pull that one SKILL.md's full text into context on demand.

So the model "knows what it could know" without paying for all of it.

Two tools:
  list_skills()       -- the cheap index (name + one-line summary each).
  load_skill(name)    -- the full SKILL.md body for one skill.

SKILLS_DIR may not exist (it's optional). Everything degrades gracefully
to "(no skills available)" rather than erroring -- a fresh sandbox has no
skills, and that's fine.
"""

def _skill_dir(name: str) -> Optional[Path]:
    """Resolve a skill folder by name, guarding against path escapes."""
    if not name or "/" in name or ".." in name:
        return None
    d = (SKILLS_DIR / name).resolve()
    try:
        d.relative_to(SKILLS_DIR.resolve())
    except ValueError:
        return None  # escaped SKILLS_DIR
    return d if (d / "SKILL.md").is_file() else None


def run_list_skills() -> str:
    """One-line-per-skill index: `name: <first non-empty line of SKILL.md>`."""
    if not SKILLS_DIR.is_dir():
        log.debug(f"[skill] no skills dir at {SKILLS_DIR}")
        return "(no skills available)"
    entries = []
    for child in sorted(SKILLS_DIR.iterdir()):
        md = child / "SKILL.md"
        if not md.is_file():
            continue
        # First non-empty line is the skill's summary (strip markdown heading marks).
        summary = ""
        for line in md.read_text(encoding="utf-8", errors="replace").splitlines():
            if line.strip():
                summary = line.lstrip("# ").strip()
                break
        entries.append(f"- {child.name}: {summary}")
    if not entries:
        return "(no skills available)"
    out = "\n".join(entries)
    log.info(f"[skill] listed {len(entries)} skill(s)")
    return out


def run_load_skill(name: str) -> str:
    """Return the full SKILL.md text for one skill, or an error string."""
    log.info(f"[skill] load {name!r}")
    d = _skill_dir(name)
    if d is None:
        avail = run_list_skills()
        return f"Error: no skill named {name!r}. Available:\n{avail}"
    body = (d / "SKILL.md").read_text(encoding="utf-8", errors="replace")
    log.debug(f"[skill] {name} -> {len(body)} chars")
    return _truncate(body)


def skills_index_for_prompt() -> str:
    """
    Compact index injected into the lead system prompt. Lets the model know
    which skills exist so it can decide when to call load_skill.
    """
    idx = run_list_skills()
    if idx == "(no skills available)":
        return ""
    return ("\n\nAvailable skills (call load_skill(name) to read the full guide "
            "when relevant):\n" + idx)


# --- Register on the LEAD registry ------------------------------------------
# Skills are a lead-agent concern (the lead decides strategy). Subagents get
# a concrete prompt already, so they don't browse the skill library.
TOOLS_LEAD += [
    _fn(
        "list_skills",
        "List available skill guides (name + one-line summary each).",
        {},
        [],
    ),
    _fn(
        "load_skill",
        "Load the full text of one skill guide by name. Call when a skill from "
        "list_skills looks relevant to the current task.",
        {"name": {"type": "string", "description": "Skill folder name from list_skills."}},
        ["name"],
    ),
]

DISPATCH_LEAD.update({
    "list_skills": lambda a: run_list_skills(),
    "load_skill":  lambda a: run_load_skill(a["name"]),
})

log.info(f"Skill loading registered. Lead registry: {list(DISPATCH_LEAD)}")


In [ ]:
"""
chat() -- the user-facing test interface.

Everything above defined capabilities. This cell wires them into a single
function you call from another cell:

    chat("list the python files in the sandbox and summarise each")

What it does, per call:
  1. (first call only) build the lead SYSTEM prompt, folding in the live
     skills index, and seed the persistent HISTORY with it.
  2. append the user's query to HISTORY.
  3. run agent_loop with the FULL lead toolset (TOOLS_LEAD / DISPATCH_LEAD)
     and the lead model -- the loop mutates HISTORY in place.
  4. after the turn, call maybe_compress(HISTORY) so a long session stays
     within the context budget.
  5. return the final answer string (also printed for convenience).

HISTORY is module-level and persists across calls, so a sequence of
chat() calls is one continuous conversation. Call reset_chat() to wipe it
and start fresh (also clears the on-disk memory/summary).

This is intentionally the *only* stateful, user-facing entry point --
mirroring how you'd `import` and call one function from a real harness.
"""

LEAD_SYSTEM = (
    f"You are a capable coding assistant operating in a sandbox at {WORKDIR}.\n"
    "You have tools for files (read, write, grep, glob, revert), shell (bash), "
    "planning (todo_write/read/update for the current request; task_create/list/"
    "update/next for durable multi-step work), delegation (spawn_subagent for "
    "noisy or self-contained subtasks), and knowledge (list_skills/load_skill).\n\n"
    "Operating principles:\n"
    "- For any multi-step request, call todo_write first to lay out a plan, then "
    "work through it, updating statuses as you go.\n"
    "- Delegate noisy exploration to a subagent so your own context stays clean.\n"
    "- Prefer reading before writing; use revert if a write was wrong.\n"
    "- When you have fully answered, reply with a clear final message and STOP "
    "calling tools -- that message is what the user sees."
)

# Persistent conversation state across chat() calls.
HISTORY: List[Dict[str, Any]] = []


def _ensure_history() -> None:
    """Seed HISTORY with the system prompt (incl. live skills index) once."""
    if not HISTORY:
        system = LEAD_SYSTEM + skills_index_for_prompt()
        HISTORY.append({"role": "system", "content": system})
        log_chat.info(f"history seeded (system prompt {len(system)} chars)")


def reset_chat() -> None:
    """Wipe the conversation and any persisted memory -- start fresh."""
    HISTORY.clear()
    try:
        MEMORY_FILE.unlink(missing_ok=True)
    except OSError:
        pass
    log_chat.info("chat history reset")


def chat(query: str) -> str:
    """
    Drive the agent for one user query. Returns (and prints) the final answer.
    HISTORY persists across calls -- this is a continuous conversation.
    """
    _ensure_history()
    log_chat.info(f"=== USER: {query[:200]!r} ===")
    HISTORY.append({"role": "user", "content": query})

    answer = agent_loop(
        messages=HISTORY,
        tools=TOOLS_LEAD,
        dispatch=DISPATCH_LEAD,
        model=MODELS["lead"],
        label="lead",
    )

    # Keep the session within budget for the *next* turn.
    compressed = maybe_compress(HISTORY)
    if compressed:
        log_chat.info("history compacted after this turn")

    log_chat.info(f"=== ASSISTANT ({len(answer)} chars) ===")
    print("\n" + "=" * 70 + "\nASSISTANT:\n" + "=" * 70)
    print(answer)
    return answer


log.info("chat() interface ready. Call chat('...') to drive the agent; "
         "reset_chat() to start over.")


In [ ]:
"""
Smoke test -- drive the whole pipeline end to end.

This is the cell you actually RUN to confirm everything wired together
works against your live Ollama. It exercises the full stack in one shot:

    healthcheck -> chat() -> agent_loop -> ollama_chat -> tool dispatch
                -> file write in the sandbox -> read-back verification

We ask for a small, deterministic, multi-step task so the run is fast and
the result is easy to verify by hand:

    1. write a file (exercises the `write` tool + snapshot)
    2. read it back  (exercises the `read` tool)
    3. report        (exercises terminal-text termination)

Because the lead system prompt nudges multi-step requests through
todo_write first, you should see todo_* calls in the logs too -- so this
one query lights up planning + tools + the loop together.

Re-runnable: we reset_chat() first so repeated runs start clean. The
assertion at the end checks the file truly landed on disk (i.e. the agent
didn't just *say* it did) -- the real test of an agent is the side effect,
not the prose.
"""

# 0) Make sure the server + models are actually reachable before we spend time.
assert ollama_healthcheck(), "Ollama not reachable / models missing -- fix config cell first."

# 1) Clean slate so this cell is idempotent.
reset_chat()

# 2) A small, verifiable, multi-step request.
SMOKE_FILE = "smoke_test.txt"
SMOKE_TEXT = "hello from the agent"
answer = chat(
    f"Create a file called {SMOKE_FILE} in the sandbox containing exactly the "
    f"text '{SMOKE_TEXT}'. Then read it back to confirm, and tell me what it contains."
)

# 3) Verify the SIDE EFFECT, not the model's prose. This is the real assertion:
#    did the file actually land on disk with the right content?
written = WORKDIR / SMOKE_FILE
print("\n" + "-" * 70)
if written.is_file():
    got = written.read_text(encoding="utf-8").strip()
    ok = SMOKE_TEXT in got
    print(f"file exists: {written}")
    print(f"contents   : {got!r}")
    print(f"smoke test : {'PASS' if ok else 'FAIL (content mismatch)'}")
else:
    print(f"smoke test : FAIL -- {written} was never created")

print("-" * 70)
print(f"history now holds {len(HISTORY)} messages "
      f"(~{_estimate_size(HISTORY)} chars).")
